In [ ]:
# Phase 4/7 (messy source, finalized): TSA weekly checkpoint throughput,
# Jan-Aug 2026 (35 weeks, matching the rebank diff-in-diff window -- see
# ADR 0003). Parsed locally with hubops.tsa_pdf (same carry-forward logic
# documented there: carried-forward date/hour/airport cells, numeric
# checkpoint names, independent date/hour carry-forward) rather than in
# Fabric -- see ADR 0008 for why (%pip is disabled for batch jobs, and
# the parsing itself is single-threaded driver work regardless of where
# it runs, so nothing was lost by doing it locally).

from pyspark.sql.functions import col, current_timestamp, lit, to_date
from pyspark.sql.types import IntegerType

WORKSPACE_ID = "16b2a9c6-6418-485f-865f-c47442b8b6c6"
LH_BRONZE_ID = "09a8b1f5-f33c-4eed-912d-40b9cbdc9b06"
ABFSS_ROOT = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LH_BRONZE_ID}"

csv_path = f"{ABFSS_ROOT}/Files/tsa/tsa_checkpoint_hour_all.csv"
df = spark.read.option("header", True).csv(csv_path)
print(f"read {df.count():,} rows from {csv_path}")


In [ ]:
df = df.withColumn("date", to_date(col("date")))
df = df.withColumn("pax", col("pax").cast(IntegerType()))
df = (
    df.withColumn("_source", lit("tsa_foia_weekly_throughput"))
    .withColumn("_loaded_at", current_timestamp())
)
df.printSchema()


In [ ]:
table_path = f"{ABFSS_ROOT}/Tables/tsa_checkpoint_hour"
df.write.format("delta").partitionBy("week_start").mode("overwrite").option("overwriteSchema", "true").save(table_path)
print(f"wrote {table_path}")


In [ ]:
check = spark.read.format("delta").load(table_path)
print(f"tsa_checkpoint_hour row count: {check.count():,}")
check.groupBy("week_start").count().orderBy("week_start").show(40)
